In [1]:
import torch
import pandas as pd
import random
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, BitsAndBytesConfig, BertConfig
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, LoftQConfig
from torch.utils.tensorboard import SummaryWriter
from transformers.integrations.integration_utils import TensorBoardCallback

# Set the random seed for reproducibility
def set_seed(seed=42): # As per Hitchhiker's Guide to the Galaxy, 42 is the answer to life, the universe, and everything
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed() 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data = pd.read_csv('./data/imdb_data.csv')

training_set = Dataset.from_pandas(data[data['dataset'] == 'train'])
test_set = Dataset.from_pandas(data[data['dataset'] == 'test'])

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # this tokenizer will work for our smaller model

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_training_set = training_set.map(tokenize_function, batched=True)
tokenized_test_set = test_set.map(tokenize_function, batched=True)

training_args = TrainingArguments(
    output_dir="./temp_results",
    num_train_epochs=3,
    per_device_train_batch_size=12,
    per_device_eval_batch_size=12,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none", # Do not report to any logging or tracking system. Logging to be done later using TensorBoard. 
        #Fix needed due to missing logging_dir parameter in TrainingArguments for newer versions of the Transformers library.
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="no",
)

W1020 17:27:31.317000 13216 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

Map:   0%|          | 0/1250 [00:00<?, ? examples/s]

In [2]:
# Configuration for 4-bit QLoRA quantization using BitsAndBytesConfig

qconfig = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    llm_int8_skip_modules=["classifier"],
)

In [3]:
# Setting up the BERT model with QLoRA quantization and LoRA configuration

model_id = "prajjwal1/bert-tiny"
bert_config = BertConfig.from_pretrained(model_id) # Load the BERT configuration from the pretrained model
bert_config.num_labels = 2

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    config=bert_config,
    quantization_config=qconfig,
    device_map={"": device},
)
loftq_config = LoftQConfig(loftq_bits=4) # Configure LoftQ for 4-bit quantization
model = prepare_model_for_kbit_training(model) # Prepare the model for k-bit (4-bit) training

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [4]:
# Configuration for LoRA (Low-Rank Adaptation) with QLoRA quantization

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # types here: https://github.com/huggingface/peft/blob/v0.7.1/src/peft/utils/peft_types.py#L22
    inference_mode=False,  # false in training, true when inferring
    loftq_config=loftq_config, # Apply the LoftQ configuration for 4-bit quantization
    r=16,  # Rank of low-rank matrices
    lora_alpha=32,  # Scaling factor for the LoRA layers
    lora_dropout=0.1  # Dropout rate, helps prevent overfitting
)

c:\Users\nomadant\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\peft\tuners\lora\config.py:1065: UserWarning: `loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.
  warnings.warn("`loftq_config` specified but will be ignored when `init_lora_weights` is not 'loftq'.")


In [5]:
# Setting up the QLoRA model with the LoRA configuration

qlora_model = get_peft_model(model, peft_config)

# Print the number of trainable parameters in the QLoRA model

def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

print_trainable_parameters(qlora_model)

trainable params: 16642 || all params: 4198020 || trainable%: 0.3964249813007084


In [6]:
# Training the QLoRA model with the Trainer API and TensorBoard integration

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized_training_set,
    eval_dataset=tokenized_test_set,
    callbacks=[TensorBoardCallback(tb_writer=SummaryWriter(log_dir="./logs"))]
)

trainer.train() # Train the model
trainer.evaluate() # Evaluate the model on the test set


c:\Users\nomadant\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\_dynamo\eval_frame.py:1548: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.691466
20,0.696609
30,0.688450
40,0.692873
50,0.697726
60,0.682618
70,0.705172
80,0.693311
90,0.688623
100,0.700270


Training Loss,Validation Loss,Step
0.689497,0.688370,315


{'eval_loss': 0.6883700489997864}

In [7]:
qlora_model.save_pretrained("./qlora_finetunedmodel")